In [ ]:
dataset_url = ""
dataset_base64 = ""
n_components = 2
target_column = ""

In [ ]:
import pandas as pd
import numpy as np
import json, io, base64, warnings
warnings.filterwarnings('ignore')
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder

result = {}
try:
    # ── Load data ──────────────────────────────────────────────────────────────
    if dataset_url:
        ext = dataset_url.split('.')[-1].lower()
        df = (pd.read_excel(dataset_url) if ext in ['xlsx', 'xls']
              else pd.read_json(dataset_url) if ext == 'json'
              else pd.read_csv(dataset_url))
    elif dataset_base64:
        raw = base64.b64decode(dataset_base64)
        try:
            df = pd.read_csv(io.BytesIO(raw))
        except Exception:
            df = pd.read_excel(io.BytesIO(raw))
    else:
        from sklearn.datasets import load_iris
        iris = load_iris()
        df = pd.DataFrame(iris.data, columns=iris.feature_names)
        df['species'] = [iris.target_names[t] for t in iris.target]
        if not target_column:
            target_column = 'species'

    # ── Separate label column ──────────────────────────────────────────────────
    label_series = None
    if target_column and target_column in df.columns:
        label_series = df[target_column].astype(str)
        df_features = df.drop(columns=[target_column])
    else:
        df_features = df.copy()

    df_num = df_features.select_dtypes(include=[np.number])
    for col in df_num.columns:
        if df_num[col].dtype in ['float64', 'int64', 'float32', 'int32']:
            df_num[col] = df_num[col].fillna(df_num[col].median())
        else:
            df_num[col] = df_num[col].fillna(df_num[col].mode()[0] if not df_num[col].mode().empty else 0)
    if df_num.shape[1] < 2:
        raise ValueError('Need at least 2 numeric columns for PCA')

    n_comp = min(int(n_components), df_num.shape[1], df_num.shape[0])

    # ── Run PCA ────────────────────────────────────────────────────────────────
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_num)
    pca = PCA(n_components=n_comp)
    X_pca = pca.fit_transform(X_scaled)

    exp_var = [round(float(v), 4) for v in pca.explained_variance_ratio_]
    cum_var = [round(float(v), 4) for v in np.cumsum(pca.explained_variance_ratio_)]

    n_plot = min(200, len(X_pca))
    pc1 = X_pca[:n_plot, 0]
    pc2 = X_pca[:n_plot, 1] if n_comp > 1 else np.zeros(n_plot)

    if label_series is not None:
        labels_arr = label_series.values[:n_plot]
        components_plot = [
            {'pc1': round(float(pc1[i]), 4), 'pc2': round(float(pc2[i]), 4), 'label': str(labels_arr[i])}
            for i in range(n_plot)
        ]
    else:
        components_plot = [
            {'pc1': round(float(pc1[i]), 4), 'pc2': round(float(pc2[i]), 4), 'label': ''}
            for i in range(n_plot)
        ]

    loadings = []
    for j, feat in enumerate(df_num.columns):
        entry = {'feature': str(feat), 'pc1': round(float(pca.components_[0, j]), 4)}
        if n_comp > 1:
            entry['pc2'] = round(float(pca.components_[1, j]), 4)
        loadings.append(entry)

    result = {
        'algorithm': 'pca',
        'n_components': n_comp,
        'explained_variance': exp_var,
        'cumulative_variance': cum_var,
        'components_plot': components_plot,
        'loadings': loadings,
    }

except Exception as e:
    import traceback
    result = {'error': str(e), 'traceback': traceback.format_exc()}

with open('/tmp/ml_result.json', 'w') as f:
    json.dump(result, f)
print(json.dumps(result, indent=2))